In [1]:
import os
import sys

# Find project root (directory containing src/ and tutorial/)
cwd = os.path.abspath(os.getcwd())
for candidate in [cwd, os.path.dirname(cwd), os.path.dirname(os.path.dirname(cwd))]:
    if os.path.isdir(os.path.join(candidate, "src")) and os.path.isdir(os.path.join(candidate, "tutorial")):
        project_root = candidate
        break
else:
    project_root = os.path.dirname(cwd)  # assume notebook in tutorial/

src_path = os.path.join(project_root, "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

# Path to the Ca triplet Zarr grid
zarr_path = os.path.join(project_root, "tutorial", "data", "regular_ca_intensity.zarr")

import logging
logging.basicConfig(level=logging.INFO)

In [2]:
import zarr

In [3]:
o = zarr.open_group(zarr_path)
o.tree()

/
├── continuum (2310, 40001) float32
├── flux (2310, 40001) float32
├── model_id (2310,) uint64
├── mu_selected (2310,) float32
├── mu_selected_index (2310,) int16
├── param_names (10,) <U32
├── params (2310, 10) float32
├── physics_hash () <U64
├── provenance
│   ├── atmosphere_manifest.json () object
│   ├── canonical_config.yaml () object
│   ├── environment.txt () object
│   ├── linelist_manifest.json () object
│   ├── software_manifest.json () object
│   └── synthesis_config.yaml () object
├── schema_version () <U16
└── wavelength (40001,) float32

# LazyZarrGridLoader Example

`LazyZarrGridLoader` lazily queries and reads spectra from a SPICE-style Zarr grid. It keeps only small metadata in memory and streams parameter scans chunk-by-chunk.

In [4]:
import zarr
import jax.numpy as jnp

from spice.spectrum.zarr_grid_interpolator import ZarrGridInterpolator
em = ZarrGridInterpolator(zarr_path, chunk_rows=2500)

INFO:2026-03-03 15:13:37,599:jax._src.xla_bridge:822: Unable to initialize backend 'tpu': INTERNAL: Failed to open libtpu.so: dlopen(libtpu.so, 0x0001): tried: 'libtpu.so' (no such file), '/System/Volumes/Preboot/Cryptexes/OSlibtpu.so' (no such file), '/Users/mjablons/anaconda3/envs/astro/bin/../lib/libtpu.so' (no such file), '/usr/lib/libtpu.so' (no such file, not in dyld cache), 'libtpu.so' (no such file)
INFO:jax._src.xla_bridge:Unable to initialize backend 'tpu': INTERNAL: Failed to open libtpu.so: dlopen(libtpu.so, 0x0001): tried: 'libtpu.so' (no such file), '/System/Volumes/Preboot/Cryptexes/OSlibtpu.so' (no such file), '/Users/mjablons/anaconda3/envs/astro/bin/../lib/libtpu.so' (no such file), '/usr/lib/libtpu.so' (no such file, not in dyld cache), 'libtpu.so' (no such file)
INFO:PASSBANDS:initializing passband (headers only) at /Users/mjablons/anaconda3/envs/astro/lib/python3.11/site-packages/phoebe/atmospheres/tables/passbands/johnson_v.fits
INFO:PASSBANDS:initializing passban

Connection to online passbands at https://tables.phoebe-project.org could not be established.  Check your internet connection or try again later (can manually call phoebe.list_online_passbands(refresh=True) to retry).  If the problem persists and you're using a Mac, you may need to update openssl (see https://phoebe-project.org/help/faq). Original error from urlopen: TimeoutError The read operation timed out


INFO:spice.spectrum.zarr_grid_loader:Opening zarr store at /Users/mjablons/Documents/spice/tutorial/data/regular_ca_intensity.zarr
INFO:spice.spectrum.zarr_grid_loader:Store opened successfully
INFO:spice.spectrum.zarr_grid_loader:Loading param_names...
INFO:spice.spectrum.zarr_grid_loader:Loaded 10 param names: ['teff', 'logg', 'feh', 'vmicro', 'a', 'c', 'n', 'o', 'r', 's']
INFO:spice.spectrum.zarr_grid_loader:Loading flux: shape=(2310, 40001), dtype=float32, requested_chunk_rows=2500, zarr_row_chunk=32, effective_chunk_rows=2310
INFO:spice.spectrum.zarr_grid_loader:Loading flux rows 0-2310 of 2310 (100%)...
INFO:spice.spectrum.zarr_grid_loader:Flux loaded: shape=(2310, 40001), dtype=float32, size=369.61 MB in 0.50s (740.89 MB/s)
INFO:spice.spectrum.zarr_grid_loader:Loading continuum: shape=(2310, 40001), dtype=float32, requested_chunk_rows=2500, zarr_row_chunk=32, effective_chunk_rows=2310
INFO:spice.spectrum.zarr_grid_loader:Loading continuum rows 0-2310 of 2310 (100%)...
INFO:spice

In [5]:
import matplotlib.pyplot as plt

%matplotlib inline

In [6]:
em.stellar_parameter_names

['logg', 'teff', 'feh', 'vmicro']